In [1]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize

def calculate_ideal_weights(
                returns,
                asset_names=None, 
                max_alloc=1, 
                max_risk=0.15, 
                risk_type='std'):
    """
    Calculate ideal portfolio weights for the next period based on historical returns.
    
    Parameters:
    -----------
    returns : numpy.ndarray
        Array of historical returns with shape (n_periods, n_assets)
    asset_names : list, optional
        List of asset names corresponding to columns in returns
    max_alloc : float, default=0.3
        Maximum allocation for any single asset (0.3 = 30%)
    max_risk : float, default=0.15
        Maximum risk allowed in the portfolio (0.15 = 15%)
    risk_type : str, default='variance'
        Type of risk measure to use ('variance', 'std', 'mad')
        
    Returns:
    --------
    pd.DataFrame
        Portfolio weights for each asset
    """
    # Calculate expected returns
    mu = np.mean(returns, axis=0)
    n_assets = returns.shape[1]
    
    # Set bounds for each asset (0 to max_alloc)
    bounds = [(0, max_alloc)] * n_assets
    
    # Set initial weights (equal allocation)
    initial_weights = np.array([1] + [0 ]* (n_assets - 1))
    
    # Define risk function based on risk_type
    if risk_type == 'std':
        # Standard deviation risk
        cov_matrix = np.cov(returns, rowvar=False)
        risk_fn = lambda w: np.sqrt(np.dot(w.T, np.dot(cov_matrix, w)))
        max_risk = max_risk * np.sqrt(1 / 252)

    elif risk_type == 'mad':
        # Mean Absolute Deviation risk
        def mad_risk(weights):
            portfolio_returns = np.dot(returns, weights)
            mean_return = np.mean(portfolio_returns)
            mad = np.mean(np.abs(portfolio_returns - mean_return))
            return mad
        risk_fn = mad_risk
        max_risk = np.sqrt(2 / np.pi) * max_risk * np.sqrt(1 / 252)
    else:
        raise ValueError(f"Unknown risk type: {risk_type}")
    
    # Define objective function (maximize expected return)
    objective_fn = lambda w: -np.dot(w, mu)
    
    # Define constraints
    constraints = [
        # Sum of weights = 1
        {'type': 'eq', 'fun': lambda w: np.sum(w) - 1},
        # Risk constraint - risk_fn(w) must be less than or equal to max_risk
        {'type': 'ineq', 'fun': lambda w: max_risk - risk_fn(w)}
    ]
    
    # Perform optimization
    result = minimize(
        objective_fn,
        initial_weights,
        method='SLSQP',
        bounds=bounds,
        constraints=constraints
    )
    
    if not result.success:
        # Print diagnostic information when optimization fails
        w = result.x
        current_risk = risk_fn(w)
        raise ValueError(f"Optimization did not converge. Current risk is {current_risk:.6f}, max allowed risk is {max_risk:.6f}.")
    
    # Create output DataFrame
    if asset_names is None:
        asset_names = [f'Asset_{i}' for i in range(n_assets)]
    
    weights = pd.DataFrame(
        result.x.reshape(-1, 1),
        index=asset_names,
        columns=['Weight']
    )
    
    return weights

In [2]:
import numpy as np

sample_returns = np.array([
    [0.0150, 0.0300],
    [0.0100, 0.0200],
    [0.0150, -0.0100],
    [0.0200, 0.0250],
    [0.0400, 0.0450],
])
asset_names = ['X', 'Y']
max_risk = 0.20

In [3]:
# Calculate ideal weights
ideal_weights = calculate_ideal_weights(
    returns=sample_returns,
    asset_names=asset_names,
    max_risk=max_risk,      # 15% maximum risk
    risk_type='mad'     # Use Mean Absolute Deviation as risk measure
)

print("Ideal Portfolio Weights:")
print(ideal_weights)

# Check that weights sum to 1
print(f"\nSum of weights: {ideal_weights['Weight'].sum():.4f}")

Ideal Portfolio Weights:
     Weight
X  0.466789
Y  0.533211

Sum of weights: 1.0000


In [4]:
# Calculate ideal weights
ideal_weights = calculate_ideal_weights(
    returns=sample_returns,
    asset_names=asset_names,
    max_risk=max_risk,      # 15% maximum risk
    risk_type='std'     
)

print("Ideal Portfolio Weights:")
print(ideal_weights)

# Check that weights sum to 1
print(f"\nSum of weights: {ideal_weights['Weight'].sum():.4f}")

Ideal Portfolio Weights:
     Weight
X  0.752261
Y  0.247739

Sum of weights: 1.0000
